In [1]:
import sys
import os


sys.path.append('../..')
os.chdir('..')

In [ ]:
# load original model
import os
import torch

from starry.utils.config import Configuration
from starry.utils.dataset_factory import loadDataset
from starry.utils.model_factory import loadModel


torch.set_printoptions(profile="full")

DATA_DIR = os.getenv('DATA_DIR')

config = Configuration.create('./configs/paraff-visionund-test.yaml', volatile=True)
train, = loadDataset(config, data_dir=DATA_DIR, splits='*0/1', device='cuda')
model = loadModel(config['model'], postfix='Loss').cuda()

it = iter(train)
batch = next(it)
batch['input_ids']

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.
2025-03-18 11:09:39.160791: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-18 11:09:39.160831: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-18 11:09:39

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

tensor([[100000,   2054,    418,    245,   9394,   4706,    285,  10046,  20308,
             13,   1257,    418,   2249,    276,   2579,    254,   7959,   3093,
            344,    254,   2677,   4614,     11,    285,   4750,    254,   2677,
            366,    245,   6265,    280,   9224,   1244,   3892,   4706,     13,
            185,    185, 100601,     25,    207, 100016, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100

In [1]:
# load trained model
import os
import torch

from starry.utils.config import Configuration
from starry.utils.dataset_factory import loadDataset
from starry.utils.model_factory import loadModel


#torch.set_printoptions(profile="full")

DATA_DIR = os.getenv('DATA_DIR')

config = Configuration(os.path.expanduser('/root/training/p/deepslog/paraff/20250327.b-paraff-visionund-janus-lycc0324-l4+norm-bf16'), volatile=True)
train, = loadDataset(config, data_dir=DATA_DIR, splits='*0/1', device='cuda')
model = loadModel(config['model'], postfix='Loss').cuda()

checkpoint = torch.load(config.localPath(config['best']), map_location='cuda')
model.deducer.load_state_dict(checkpoint['model'])

model.eval()

# neccesary for use 'janus.generate'
model.deducer.merge_embeddings()

it = iter(train)

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [13]:
batch = next(it)
batch['input_ids']

tensor([[100000,   2054,    418,  ..., 100015, 100015, 100015],
        [100000,   2054,    418,  ..., 100001, 100015, 100015],
        [100000,   2054,    418,  ..., 100015, 100015, 100015],
        ...,
        [100000,   2054,    418,  ...,    207, 100606, 100001],
        [100000,   2054,    418,  ..., 100015, 100015, 100015],
        [100000,   2054,    418,  ..., 100001, 100015, 100015]],
       device='cuda:0')

In [14]:
prompt_len = batch['target_mask'][0].tolist().index(True) + 1
prompt_len

637

In [15]:
input_ids = batch['input_ids'][:, :prompt_len]
input_ids

tensor([[100000,   2054,    418,  ...,  20511,     13,    207],
        [100000,   2054,    418,  ...,  20511,     13,    207],
        [100000,   2054,    418,  ...,    207, 100605,    207],
        ...,
        [100000,   2054,    418,  ...,  20511,     13,    207],
        [100000,   2054,    418,  ...,  20511,     13,    207],
        [100000,   2054,    418,  ...,    207, 100605,    207]],
       device='cuda:0')

In [16]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("/root/training/jp7")

input_text = tokenizer.decode(input_ids[0])
input_text

'<｜begin▁of▁sentence｜>You are a helpful language and vision assistant. You are able to understand the visual content that the user provides, and assist the user with a variety of tasks using natural language.\n\n<|User|>: <begin_of_image><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><i

In [17]:
with torch.no_grad():
	img_emb = model.aligner(batch['img_emb'][:1].to(model.dtype))
	img_emb.shape

In [18]:
inputs_embeds = model.deducer.embed_input(input_ids[:1])
image_seq_mask = batch['image_seq_mask']
img_emb = img_emb.reshape((-1, img_emb.shape[-1]))
inputs_embeds[image_seq_mask[:1, :prompt_len]] = img_emb
inputs_embeds.shape

torch.Size([1, 637, 4096])

In [19]:
with torch.no_grad():
	outputs = model.deducer.janus.generate(
		inputs_embeds=inputs_embeds,
		#attention_mask=batch['attention_mask'][:1, :prompt_len],
		pad_token_id=tokenizer.eos_token_id,
		bos_token_id=tokenizer.bos_token_id,
		eos_token_id=tokenizer.eos_token_id,
		max_new_tokens=64,
		do_sample=False,
		use_cache=True,
		temperature=0,
		top_p=1,
	)

outputs

tensor([[100605,    207, 100611,    207, 100627,    207, 100637,    207, 100607,
            207, 100610,    207,     64,    207, 100640,    207, 100645,    207,
          14433,    207, 100606, 100001]], device='cuda:0')

In [20]:
ie = inputs_embeds

for i in range(64):
	logits = model.deducer.janus(inputs_embeds=ie).logits
	new_id = logits[:, -1].argmax(dim=-1)

	print(i, new_id, tokenizer.decode(new_id))

	new_emb = model.deducer.embed_input(new_id[:, None])
	ie = torch.cat([ie, new_emb], dim=1)


0 tensor([100605], device='cuda:0') BOM
1 tensor([207], device='cuda:0')  
2 tensor([100611], device='cuda:0') K0
3 tensor([207], device='cuda:0')  
4 tensor([100627], device='cuda:0') TN4
5 tensor([207], device='cuda:0')  
6 tensor([100637], device='cuda:0') TD4


7 tensor([207], device='cuda:0')  
8 tensor([100607], device='cuda:0') S1
9 tensor([207], device='cuda:0')  
10 tensor([100610], device='cuda:0') Cg
11 tensor([207], device='cuda:0')  
12 tensor([64], device='cuda:0') a
13 tensor([207], device='cuda:0')  
14 tensor([100640], device='cuda:0') Osup
15 tensor([207], device='cuda:0')  
16 tensor([100645], device='cuda:0') D1
17 tensor([207], device='cuda:0')  
18 tensor([14433], device='cuda:0') Rest
19 tensor([207], device='cuda:0')  
20 tensor([100606], device='cuda:0') EOM
21 tensor([100001], device='cuda:0') <｜end▁of▁sentence｜>
22 tensor([100000], device='cuda:0') <｜begin▁of▁sentence｜>
23 tensor([100001], device='cuda:0') <｜end▁of▁sentence｜>
24 tensor([100000], device='cuda:0') <｜begin▁of▁sentence｜>
25 tensor([207], device='cuda:0')  
26 tensor([100645], device='cuda:0') D1
27 tensor([207], device='cuda:0')  
28 tensor([14433], device='cuda:0') Rest
29 tensor([207], device='cuda:0')  
30 tensor([100606], device='cuda:0') EOM
31 tensor(

In [21]:
tokenizer.decode(outputs[0])

'BOM K0 TN4 TD4 S1 Cg a Osup D1 Rest EOM<｜end▁of▁sentence｜>'

In [22]:
target_ids = batch['input_ids'][:, prompt_len:]
target_ids

tensor([[100605,    207, 100611,    207, 100627,    207, 100637,    207, 100607,
            207, 100610,    207,     64,    207, 100640,    207, 100645,    207,
          14433,    207, 100606, 100001, 100015, 100015, 100015, 100015, 100015,
         100015, 100015, 100015, 100015, 100015, 100015, 100015, 100015, 100015,
         100015, 100015, 100015, 100015],
        [100605,    207, 100619,    207, 100626,    207, 100637,    207, 100607,
            207, 100610,    207,  40647,    207,     70,    207, 100640,    207,
         100647,    207,  40149,    207, 100681,    207,     64,    207, 100648,
            207,  82708,    207,     65,    207,  27841,    207, 100647,    207,
         100606, 100001, 100015, 100015],
        [100625,    207, 100637,    207, 100607,    207, 100610,    207,  82708,
            207,     66,    207,  27841,    207, 100640,    207, 100647,    207,
             65,    207,  27841,    207, 100648,    207,   7643,    207,     68,
            207,  27841, 

In [23]:
target_text = tokenizer.decode(target_ids[0])
target_text

'BOM K0 TN4 TD4 S1 Cg a Osup D1 Rest EOM<｜end▁of▁sentence｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜><｜▁pad▁｜>'